In [1]:
import json
import os


In [ ]:
path = "annotations.json"
with open(path, "r") as f:
    data = json.load(f)
    l_predicate = []
    print(len(data))
    for datapoint in data:
        for example in datapoint["annotations"]:
            l_predicate.append(example["predicate"])

set_predicate = set(l_predicate)

print("Number of unique predicates: ", len(set_predicate))
print("Unique predicates: ", set_predicate)

11569
Number of unique predicates:  9
Unique predicates:  {'on', 'to the right of', 'under', 'in', 'in front of', 'above', 'next to', 'behind', 'to the left of'}


In [35]:
import json
from PIL import Image, ImageDraw, ImageFont
import argparse
import random
import os
import sys
import pdb

import datasets

home = os.path.expanduser("~")

/home/samaier/miniconda3/envs/spatialsense/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
DATA_DIR = "/scratch/izar/samaier/vlm_r1/data"
IMAGES_DIR = os.path.join(DATA_DIR, "images/SpatialSense")
SAVE_DIR = os.path.join(DATA_DIR, "SpatialSense")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

In [11]:
print(DATA_DIR)
print(IMAGES_DIR)
print(SAVE_DIR)

/scratch/izar/samaier/vlm_r1/data
/scratch/izar/samaier/vlm_r1/data/images/SpatialSense
/scratch/izar/samaier/vlm_r1/data/SpatialSense


In [29]:
dataroot = '/home/samaier/SpatialSense'

In [30]:
def url2path(url):
    if url.startswith("http"):  # flickr
        return os.path.join(dataroot, "images", "flickr", url.split("/")[-1])
    else:  # nyu
        return os.path.join(dataroot, "images", "nyu", url.split("/")[-1])

In [15]:
splits = ["train", "validation", "test"]
dataset_split = {}

In [18]:
perc_train = 0.8
perc_valtest = 0.5

In [20]:
# create a dict for each predicate and have an empty list
predicate_dataset = {}
for predicate in set_predicate:
    predicate_dataset[predicate] = []

for img in data:
    for annotation in img["annotations"]:
        predicate = annotation["predicate"]
        annotation['url'] = img["url"]
        annotation['width'] = img["width"]
        annotation['height'] = img["height"]
        predicate_dataset[predicate].append(annotation)

for split in splits:
    dataset_split[split] = []

for predicate in predicate_dataset:
    train_predcate = predicate_dataset[predicate][:int(len(predicate_dataset[predicate]) * perc_train)]
    valtest_predcate = predicate_dataset[predicate][int(len(predicate_dataset[predicate]) * perc_train):]
    val_predcate = valtest_predcate[:int(len(valtest_predcate) * perc_valtest)]
    test_predcate = valtest_predcate[int(len(valtest_predcate) * perc_valtest):]

    dataset_split["train"] += train_predcate
    dataset_split["validation"] += val_predcate
    dataset_split["test"] += test_predcate

for split in dataset_split:
    print(f"Number of {split} examples: ", len(dataset_split[split]))

Number of train examples:  13994
Number of validation examples:  1750
Number of test examples:  1754


In [38]:
hf_datasets_split = {}
for split in dataset_split:
    os.makedirs(os.path.join(IMAGES_DIR, split), exist_ok=True)
    split_data = {'subject': [], 'predicate': [], 'object': [], 'label': [], 'image_path': []}
    for i, example in enumerate(dataset_split[split]):
        # create image
        img = Image.open(url2path(example["url"]))
        vis = Image.new("RGB", (max(example["width"], 300), example["height"] + 40), color="white")
        vis.paste(img)
        draw = ImageDraw.Draw(vis)
        path_img = os.path.join(IMAGES_DIR, split, "%05d.jpg" % i)
        vis.save(path_img)

        # create json
        subject = example["subject"]['name']
        predicate = example["predicate"]
        object = example["object"]['name']
        label = str(example["label"])
        image_path = path_img

        split_data['subject'].append(subject)
        split_data['predicate'].append(predicate)
        split_data['object'].append(object)
        split_data['label'].append(label)
        split_data['image_path'].append(image_path)

    hf_dataset = datasets.Dataset.from_dict(split_data)
    hf_datasets_split[split] = hf_dataset

dataset = datasets.DatasetDict(hf_datasets_split)
dataset.save_to_disk(SAVE_DIR)